# Plot results for offline prompting in the "farm" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [2]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys
import json

In [3]:
results_folder = Path("generated_adaptations/farm")

## Run experiments

In [4]:
prompts = {
    "default": "generated_adaptations/prompts/farm_strategy.md",
    "state": "generated_adaptations/prompts/farm_strategy_state.md",
}
variants = {
    "default": [],
    "notest": ["--retries_test=0"],
    "constraints": []
}
llms = {
    # "5nano": "gpt-5-nano-2025-08-07",
    "5nanolow": "gpt-5-nano-2025-08-07,reasoning_effort=low",
}
repeats = 1
start = 1

In [5]:
# import generated_adaptations.generator as generator

In [6]:
for prompt_name, prompt in prompts.items():
    for variant, args in variants.items():
        for llm_name, llm in llms.items():
            for repeat in range(start, start + repeats):
                folder_name = f"{llm_name}_{repeat:02d}"
                print(f"\n{variant}_{prompt_name}/{folder_name}\n")
                folder = results_folder / f"{variant}_{prompt_name}" / folder_name
                folder.mkdir(parents=True, exist_ok=True)
                shutil.copy(prompt, folder / "01_01_user.md")
                cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}", f"--llm={llm}", *args]
                result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
                print(result.stdout)
                print(result.stderr)
                # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


default_default/5nanolow_01

Loaded 2 messages from generated_adaptations\farm\default_default\5nanolow_01.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\farm\default_default\5nanolow_01\code_01_02.py'.
TOKENS USED:
Input: 1010, Output: 2219 (reasoning: 896)
Response time (seconds): 13.4

Running tests: pytest generated_adaptations/tests -q --tb=short -rfExXpP --show-capture=no --color=no --example=farm --adaptation_name=5nanolow_01/code_01_02 --variant=default_default
Test exit code: 0
Running simulation for 'generated_adaptations\farm\default_default\5nanolow_01\code_01_02.py'.
  Run #1/3: python main.py farm/configs/default.yaml generated_adaptations/configs/generated.yaml farm/configs/config_no_battery.yaml DSL/drones.yaml --extra_config={"name": "default_default/5nanolow_01/code_01_02", "log_dir.append": "/default_default/5nanolow_01/code_01_02", "adaptation_name": "generated_adaptations.farm.default_default.

## Results

In [7]:
# folders = list(results_folder.glob("41mini_*"))
folders = list(results_folder.glob("*/5nanolow*"))
# folders = [results_folder / "41mini"]
print([f.stem for f in folders])

['5nanolow_01', '5nanolow_01', '5nanolow_01', '5nanolow_01', '5nanolow_01', '5nanolow_01']


In [18]:
summary = pd.DataFrame(columns=["llm", "params", "repeat"])
best = pd.DataFrame(columns=["llm", "params", "repeat"])
for folder in folders:
    llm, repeat = folder.stem.split("_")
    params = folder.parent.stem
    summary.loc[len(summary), ["llm", "params", "repeat"]] = [llm, params, repeat]
    best.loc[len(best), ["llm", "params", "repeat"]] = [llm, params, repeat]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "pass"
        elif "_simulation_result" in name:  # older: "_simulation_result.txt"
            code = name.removesuffix("_simulation_result")
            with open(file, "r") as f:
                result = f.read().strip()
                result = result.split("\n")[0].split(": ")[-1]  # note that this is specific for the farm example
            summary.loc[len(summary) - 1, code + "_result"] = result
            best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
        else:
            print(f"Unknown file: {file}")
        for file in (folder / "results").glob("*.json"):  # newer: "_simulation_result.json"
            name = file.stem.removeprefix("code_")
            if "_simulation_result" in name:
                code = name.removesuffix("_simulation_result")
                results = json.load(open(file))
                result = results["damage"]
                summary.loc[len(summary) - 1, code + "_result"] = result
                best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
            else:
                print(f"Unknown file: {file}")
summary.fillna("", inplace=True)
best.fillna("", inplace=True)

C:\Users\micha\AppData\Local\Temp\ipykernel_21996\2306563932.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary.fillna("", inplace=True)


In [19]:
summary.set_index(["llm", "params", "repeat"], inplace=True)
best.set_index(["llm", "params", "repeat"], inplace=True)

In [20]:
summary = summary.reindex(sorted(summary.columns), axis=1)
best = best.reindex(sorted(best.columns), axis=1)

In [16]:
summary

01_02_result 01_02_test 01_04_result  \
llm      params              repeat                                        
5nanolow constraints_default 01                        fail   120.666667   
         constraints_state   01       123.333333       fail        123.0   
         default_default     01        56.666667       pass                
         default_state       01                        fail        286.0   
         notest_default      01       120.666667       pass                
         notest_state        01            278.0       fail                

                                    01_04_test 01_06_result 01_06_test  \
llm      params              repeat                                      
5nanolow constraints_default 01           fail   143.666667       pass   
         constraints_state   01           fail   128.666667       pass   
         default_default     01                                          
         default_state       01           pass                           
         notest_default      01                                          
         notest_state        01                                          

                                     02_02_result 02_02_test 02_04_result  \
llm      params              repeat                                         
5nanolow constraints_default 01         74.000000       fail   120.666667   
         constraints_state   01        128.666667       pass                
         default_default     01        187.000000       pass                
         default_state       01        286.666667       pass                
         notest_default      01        241.666667       fail                
         notest_state        01        241.666667       fail                

                                    02_04_test 02_06_result 02_06_test  \
llm      params              repeat                                      
5nanolow constraints_default 01           fail   143.666667       pass   
         constraints_state   01                                          
         default_default     01                                          
         default_state       01                                          
         notest_default      01                                          
         notest_state        01                                          

                                     03_02_result 03_02_test 03_04_result  \
llm      params              repeat                                         
5nanolow constraints_default 01        312.666667       fail   143.666667   
         constraints_state   01        268.000000       fail        123.0   
         default_default     01        126.333333       pass                
         default_state       01        178.000000       fail        178.0   
         notest_default      01        123.333333       pass                
         notest_state        01        241.666667       fail                

                                    03_04_test 03_06_result 03_06_test  
llm      params              repeat                                     
5nanolow constraints_default 01           pass                          
         constraints_state   01           fail   128.666667       pass  
         default_default     01                                         
         default_state       01           fail   177.666667       pass  
         notest_default      01                                         
         notest_state        01

In [21]:
best

01_result 01_test   02_result 02_test  \
llm      params              repeat                                           
5nanolow constraints_default 01      143.666667    pass  143.666667    pass   
         constraints_state   01      128.666667    pass  128.666667    pass   
         default_default     01       56.666667    pass  187.000000    pass   
         default_state       01      286.000000    pass  286.666667    pass   
         notest_default      01      120.666667    pass  241.666667    fail   
         notest_state        01      278.000000    fail  241.666667    fail   

                                      03_result 03_test  
llm      params              repeat                      
5nanolow constraints_default 01      143.666667    pass  
         constraints_state   01      128.666667    pass  
         default_default     01      126.333333    pass  
         default_state       01      177.666667    pass  
         notest_default      01      123.333333    pass  
         notest_state        01      241.666667    fail